# EVOLVE — Test Set Evaluation and Performance Analysis

## 1. Test Set Evaluation

This notebook evaluates the best models selected in Notebook 1 on a strictly held-out test set.

The evaluation includes:

- Global detection metrics (mAP, precision, recall)
- Per-class performance
- Image-level recall computation
- Analysis of performance variability using scene descriptors
- Qualitative inspection of failure cases

In [ ]:
%matplotlib inline

In [ ]:
# ==========================================
# Experimental Setup
# ==========================================

import random
import torch
import yaml
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------
# Reproducibility parameters
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------
# Device configuration
# ------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device used:", device)
print("Torch CUDA available:", torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
print("Random seed:", SEED)

# ------------------------------
# Resolve project structure
# ------------------------------

PROJECT_ROOT = Path("..").resolve()

DATASET_CONFIG_PATH = (PROJECT_ROOT / "configs" / "dataset.yaml").resolve()

assert DATASET_CONFIG_PATH.exists(), f"Dataset config not found: {DATASET_CONFIG_PATH}"

print("Project root:", PROJECT_ROOT)
print("Dataset config path:", DATASET_CONFIG_PATH)

# ------------------------------
# Load dataset configuration
# ------------------------------

with open(DATASET_CONFIG_PATH, "r") as f:
    dataset_config = yaml.safe_load(f)

# Resolve dataset base path relative to dataset.yaml location
BASE_PATH = (DATASET_CONFIG_PATH.parent / dataset_config["path"]).resolve()

test_image_path = BASE_PATH / dataset_config["test"]
test_label_path = BASE_PATH / "labels" / "test"

assert test_image_path.exists(), f"Test image directory not found: {test_image_path}"
assert test_label_path.exists(), f"Test label directory not found: {test_label_path}"

print("Resolved dataset root:", BASE_PATH)
print("Resolved test images:", test_image_path)
print("Resolved test labels:", test_label_path)

# ------------------------------
# Class names
# ------------------------------

names = dataset_config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

print("Number of classes:", len(names))
print("Class names:", names)

In [ ]:
# ==============================
# Results directories
# ==============================

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
QUALITATIVE_DIR = RESULTS_DIR / "qualitative_examples"
CONFUSION_DIR = RESULTS_DIR / "confusion_analysis"

for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR, QUALITATIVE_DIR, CONFUSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Results directories initialized.")

## 2. Dataset

The EVOLVE dataset consists of annotated concert images collected under low-light and high-density conditions.

### Classes

- `amp`
- `guitar`
- `drums`
- `micro`
- `mosh_pit`
- `hands_raised`

A stratified multi-label split was performed to preserve class distribution across:

- Training
- Validation
- Test

The test set is strictly independent and was not used during model training or hyperparameter selection.

In [ ]:
# ==============================
# Load trained models (Pretrained vs Scratch)
# ==============================

from ultralytics import YOLO

RUNS = {
    "pretrained": "evolve_yolov8s_pretrained",
    "scratch": "evolve_yolov8s_scratch"
}

models = {}

for label, run_name in RUNS.items():
    model_path = (PROJECT_ROOT / "runs/train" / run_name / "weights" / "best.pt").resolve()
    assert model_path.exists(), f"Model not found at: {model_path}"
    models[label] = YOLO(str(model_path))
    print(f"Loaded {label} model from:", model_path)

In [ ]:
# ==============================
# Test Evaluation (Pretrained vs Scratch)
# ==============================

import pandas as pd
import shutil
from pathlib import Path

test_metrics = {}

for label, model in models.items():
    print(f"\nEvaluating {label} model on test set...")
    
    metrics = model.val(
        data=DATASET_CONFIG_PATH,
        split="test",
        device=device,
        verbose=False
    )
    
    test_metrics[label] = metrics

    print(f"\nTest Results ({label}):")
    print("mAP50-95:", float(metrics.box.map))
    print("mAP50:", float(metrics.box.map50))
    print("Precision:", float(metrics.box.mp))
    print("Recall:", float(metrics.box.mr))


# ==============================
# Global Comparison Table
# ==============================

comparison_test = pd.DataFrame([
    {
        "Model": label,
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "Precision": float(metrics.box.mp),
        "Recall": float(metrics.box.mr),
    }
    for label, metrics in test_metrics.items()
])

print("\nTest Comparison Table:")
display(comparison_test)

# Save global metrics
comparison_test.to_csv(
    TABLES_DIR / "global_metrics.csv",
    index=False
)

# Delta (Pretrained - Scratch)
delta_test = (
    comparison_test.set_index("Model").loc["pretrained"]
    - comparison_test.set_index("Model").loc["scratch"]
)

print("\nTransfer Learning Gain on TEST (Pretrained - Scratch):")
display(delta_test.to_frame(name="delta"))

delta_test.to_frame(name="delta").to_csv(
    TABLES_DIR / "delta_global_metrics.csv"
)


# ==============================
# Per-Class Results
# ==============================

# Ensure correct class order
names = dataset_config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
else:
    names = list(names)

per_class_list = []

for label, metrics in test_metrics.items():

    box = metrics.box

    per_class_df = pd.DataFrame({
        "Model": label,
        "Class": names,
        "AP50-95": list(getattr(box, "maps", [None] * len(names))),
    })

    if hasattr(box, "ap50"):
        per_class_df["AP50"] = list(box.ap50)

    if hasattr(box, "p"):
        per_class_df["Precision"] = list(box.p)

    if hasattr(box, "r"):
        per_class_df["Recall"] = list(box.r)

    per_class_list.append(per_class_df)

per_class_results = pd.concat(per_class_list, ignore_index=True)

print("\nPer-class results:")
display(per_class_results)

# Save per-class metrics
per_class_results.to_csv(
    TABLES_DIR / "per_class_metrics.csv",
    index=False
)


# ==============================
# Delta Per Class (AP50-95)
# ==============================

delta_per_class = (
    per_class_results
    .pivot(index="Class", columns="Model", values="AP50-95")
)

delta_per_class["delta_AP50-95"] = (
    delta_per_class["pretrained"] - delta_per_class["scratch"]
)

print("\nDelta per class (AP50-95):")
display(delta_per_class)

delta_per_class.to_csv(
    TABLES_DIR / "delta_per_class_AP50-95.csv"
)


# ==============================
# Export Confusion Matrices
# ==============================

for label, metrics in test_metrics.items():

    run_dir = Path(metrics.save_dir)
    src = run_dir / "confusion_matrix.png"

    if src.exists():
        shutil.copy(
            src,
            CONFUSION_DIR / f"confusion_matrix_{label}.png"
        )

print("\nTest evaluation exports completed.")

In [ ]:
# ==============================
# Utilities — YOLO label parsing + IoU
# ==============================

import numpy as np

def yolo_to_xyxy(line, img_w, img_h):
    """
    Convert one YOLO label line into (class_id, [x1, y1, x2, y2]) in pixel coordinates.

    YOLO format:
        class_id x_center y_center width height
    where coordinates are normalized in [0,1].
    """
    parts = line.strip().split()
    class_id = int(parts[0])

    x_c = float(parts[1]) * img_w
    y_c = float(parts[2]) * img_h
    w   = float(parts[3]) * img_w
    h   = float(parts[4]) * img_h

    x1 = x_c - w / 2
    y1 = y_c - h / 2
    x2 = x_c + w / 2
    y2 = y_c + h / 2

    return class_id, np.array([x1, y1, x2, y2], dtype=float)


def compute_iou(boxA, boxB):
    """
    Compute IoU between two boxes in [x1, y1, x2, y2] format (pixel coords).
    """
    xA = max(float(boxA[0]), float(boxB[0]))
    yA = max(float(boxA[1]), float(boxB[1]))
    xB = min(float(boxA[2]), float(boxB[2]))
    yB = min(float(boxA[3]), float(boxB[3]))

    inter_w = max(0.0, xB - xA)
    inter_h = max(0.0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0.0, float(boxA[2]) - float(boxA[0])) * max(0.0, float(boxA[3]) - float(boxA[1]))
    areaB = max(0.0, float(boxB[2]) - float(boxB[0])) * max(0.0, float(boxB[3]) - float(boxB[1]))

    union_area = areaA + areaB - inter_area

    if union_area <= 0:
        return 0.0

    return inter_area / union_area


# ------------------------------
# Image-level evaluation settings
# ------------------------------

IOU_THR = 0.5
PRED_CONF = 0.25
EXCLUDE_EMPTY_GT_IMAGES = True

print("Utilities loaded.")
print("IOU_THR:", IOU_THR, "| PRED_CONF:", PRED_CONF, "| EXCLUDE_EMPTY_GT_IMAGES:", EXCLUDE_EMPTY_GT_IMAGES)

In [ ]:
# ==============================
# Image-level metrics computation
# ==============================

from PIL import Image

performance_tables = []

for label, model in models.items():

    print(f"\nComputing image-level metrics for {label} model...")

    performance_data = []

    test_images = sorted(
        list(test_image_path.glob("*.jpg")) +
        list(test_image_path.glob("*.png"))
    )

    for img_path in test_images:

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        label_path = test_label_path / (img_path.stem + ".txt")

        if not label_path.exists():
            continue

        # ---- Load GT ----
        gt_boxes = []
        with open(label_path, "r") as f:
            for line in f:
                if line.strip():
                    class_id, box = yolo_to_xyxy(line, img_w, img_h)
                    gt_boxes.append((class_id, box))

        if len(gt_boxes) == 0:
            continue

        # ---- Predictions ----
        results = model.predict(img_path, conf=0.25, verbose=False)

        pred_boxes = []
        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                pred_boxes.append((
                    int(box.cls[0]),
                    box.xyxy[0].cpu().numpy()
                ))

        # ---- Matching ----
        matched_pred = set()
        tp = 0

        for gt_class, gt_box in gt_boxes:
            for j, (pred_class, pred_box) in enumerate(pred_boxes):

                if j in matched_pred:
                    continue

                if pred_class != gt_class:
                    continue

                if compute_iou(gt_box, pred_box) >= 0.5:
                    tp += 1
                    matched_pred.add(j)
                    break

        fn = len(gt_boxes) - tp
        fp = len(pred_boxes) - len(matched_pred)

        precision_i = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_i    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_i        = (
            2 * precision_i * recall_i / (precision_i + recall_i)
            if (precision_i + recall_i) > 0 else 0.0
        )

        performance_data.append({
            "Model": label,
            "image": img_path.name,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision_i,
            "recall": recall_i,
            "f1": f1_i,
        })

    df_model = pd.DataFrame(performance_data)
    performance_tables.append(df_model)

performance_df = pd.concat(performance_tables, ignore_index=True)

print("Image-level dataset shape:", performance_df.shape)
performance_df.head()

In [ ]:
# ==============================
# Save image-level analysis
# ==============================

performance_df.to_csv(
    TABLES_DIR / "image_level_analysis.csv",
    index=False
)

print("Image-level analysis exported.")

In [ ]:
# ==============================
# Best and Worst Recall Images
# ==============================

# Ensure qualitative directory exists
QUALITATIVE_DIR.mkdir(parents=True, exist_ok=True)

# Sort images by recall
sorted_df = performance_df.sort_values("recall", ascending=False)

# Select top and bottom images
best_images = sorted_df["image"].unique()[:2]
worst_images = sorted_df.sort_values("recall")["image"].unique()[:2]

print("Best recall images:")
for img in best_images:
    print(f"Visualizing: {img}")
    visualize_image(img)

print("\nWorst recall images:")
for img in worst_images:
    print(f"Visualizing: {img}")
    visualize_image(img)

print("\nQualitative examples saved to:", QUALITATIVE_DIR)

In [ ]:
# ==============================
# Export test metrics (Pretrained vs Scratch)
# ==============================

import json

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

# --- Export global comparison ---
comparison_test.to_csv(results_dir / "test_global_comparison.csv", index=False)

# --- Export delta ---
delta_test.to_frame(name="delta").to_csv(
    results_dir / "test_global_delta.csv"
)

# --- Export per-class results ---
per_class_results.to_csv(
    results_dir / "test_per_class_comparison.csv",
    index=False
)

# --- Export JSON summary ---
test_summary = {
    label: {
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "Precision": float(metrics.box.mp),
        "Recall": float(metrics.box.mr),
    }
    for label, metrics in test_metrics.items()
}

with open(results_dir / "test_metrics_summary.json", "w") as f:
    json.dump(test_summary, f, indent=4)

print("Test metrics exported successfully to:", results_dir)

## 3. Image-Level Performance Analysis

### 3.1. Image-Level Recall Definition

We define image-level recall as:

$$\text{Recall}_i = \frac{\text{TP}_i}{\text{(TP}_i + \text{FN}_i)}$$

where:
- $\text{TP}_i$ is the number of correctly detected objects in image $i$
- $\text{FN}_i$ is the number of missed objects
- A match is defined using $\text{IoU} \ge 0.5$ and correct class prediction.

Images containing no ground-truth objects are excluded from image-level analysis.

### 3.2. Visual Descriptors

For each test image, the following visual descriptors are computed:

- Mean luminance
- Normalized luminance
- Luminance variance
- Blur estimate
- Entropy

These descriptors aim to quantify illumination and image quality conditions.

In [ ]:
# ==============================
# Prepare test image and label lists
# ==============================

from PIL import Image

test_images = sorted(list(test_image_path.glob("*.jpg")))
test_labels_dir = test_label_path

print("Number of test images:", len(test_images))

In [ ]:
# ==============================
# Image-level metrics on TEST set (Pretrained vs Scratch)
# ==============================

from PIL import Image
import numpy as np

# --------------------------------------------------
# Prepare test image and label lists
# --------------------------------------------------

test_images = sorted(
    list(test_image_path.glob("*.jpg")) +
    list(test_image_path.glob("*.png"))
)

test_labels_dir = test_label_path

print("Resolved test images:", test_image_path)
print("Resolved test labels:", test_labels_dir)
print("Number of test images:", len(test_images))

# --------------------------------------------------
# Compute image-level metrics
# --------------------------------------------------

performance_tables = []

for label, model in models.items():

    print(f"\nComputing image-level metrics for {label} model...")

    performance_data = []

    for img_path in test_images:

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        label_path = test_labels_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            continue

        # --------------------------------------------------
        # Load ground-truth boxes
        # --------------------------------------------------

        with open(label_path, "r") as f:
            gt_lines = [ln.strip() for ln in f.readlines() if ln.strip()]

        gt_boxes = []

        for ln in gt_lines:
            class_id, box = yolo_to_xyxy(ln, img_w, img_h)
            gt_boxes.append((class_id, box))

        if EXCLUDE_EMPTY_GT_IMAGES and len(gt_boxes) == 0:
            continue

        # --------------------------------------------------
        # Model prediction
        # --------------------------------------------------

        pred = model.predict(
            img_path,
            device=device,
            conf=PRED_CONF,
            verbose=False
        )

        pred_boxes = []

        if pred and pred[0].boxes is not None:
            boxes = pred[0].boxes
            for i in range(len(boxes)):
                pred_class = int(boxes.cls[i].item())
                pred_xyxy = boxes.xyxy[i].cpu().numpy()
                pred_boxes.append((pred_class, pred_xyxy))

        # --------------------------------------------------
        # Matching (IoU >= threshold + correct class)
        # --------------------------------------------------

        matched_pred = set()
        tp = 0

        for gt_class, gt_box in gt_boxes:

            for j, (pred_class, pred_box) in enumerate(pred_boxes):

                if j in matched_pred:
                    continue

                if pred_class != gt_class:
                    continue

                if compute_iou(gt_box, pred_box) >= IOU_THR:
                    tp += 1
                    matched_pred.add(j)
                    break

        fn = len(gt_boxes) - tp
        fp = len(pred_boxes) - len(matched_pred)

        precision_i = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_i    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_i        = (
            2 * precision_i * recall_i / (precision_i + recall_i)
            if (precision_i + recall_i) > 0 else 0.0
        )

        performance_data.append({
            "Model": label,
            "image": img_path.name,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision_i,
            "recall": recall_i,
            "f1": f1_i,
            "num_gt_objects": len(gt_boxes),
            "num_pred_objects": len(pred_boxes),
        })

    df_model = pd.DataFrame(performance_data)
    performance_tables.append(df_model)

performance_df = pd.concat(performance_tables, ignore_index=True)

print("Total rows in image-level dataset:", len(performance_df))
print("Unique images:", performance_df["image"].nunique())
print("Models present:", performance_df["Model"].unique())

performance_df.head()

In [ ]:
pivot_recall = performance_df.pivot(index="image", columns="Model", values="recall")

In [ ]:
# ==============================
# Paired t-test on recall (Pretrained vs Scratch)
# ==============================

from scipy.stats import ttest_rel

ttest_result = ttest_rel(
    pivot_recall["pretrained"],
    pivot_recall["scratch"]
)

print("Paired t-test (recall):")
print(ttest_result)

# Save result to file
with open(TABLES_DIR / "t_test_recall_pretrained_vs_scratch.txt", "w") as f:
    f.write("Paired t-test on recall (Pretrained vs Scratch)\n")
    f.write(f"Statistic: {ttest_result.statistic}\n")
    f.write(f"P-value: {ttest_result.pvalue}\n")

print("T-test results exported.")

In [ ]:
# ==============================
# Additional imports for image descriptors
# ==============================

import cv2

try:
    from skimage.measure import shannon_entropy
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print("Warning: scikit-image not installed. Shannon entropy descriptor disabled.")

if HAS_SKIMAGE:
    import skimage
    print("scikit-image version:", skimage.__version__)

In [ ]:
# ==============================
# Compute visual descriptors
# ==============================

descriptor_data = []

# Only compute descriptors for images retained in performance_df
images_for_analysis = set(performance_df["image"].unique())

for img_path in test_images:

    if img_path.name not in images_for_analysis:
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    luminance_mean = float(np.mean(gray))
    luminance_var  = float(np.var(gray))
    luminance_norm = luminance_mean / 255.0  # normalized luminance (0–1)

    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    if HAS_SKIMAGE:
        entropy = float(shannon_entropy(gray))
    else:
        entropy = np.nan

    descriptor_data.append({
        "image": img_path.name,
        "luminance_mean": luminance_mean,
        "luminance_norm": luminance_norm,
        "luminance_var": luminance_var,
        "blur": blur,
        "entropy": entropy
    })

descriptor_df = pd.DataFrame(descriptor_data)

print("Descriptor rows:", len(descriptor_df))
descriptor_df.head()

In [ ]:
# ==============================
# Merge performance and descriptors
# ==============================

analysis_df = performance_df.merge(
    descriptor_df,
    on="image",
    how="inner"
)

print("Final analysis dataset size:", len(analysis_df))

# Sanity check
expected_rows = len(performance_df)
if len(analysis_df) != expected_rows:
    print("Warning: some rows were dropped during merge.")
else:
    print("Merge successful: no rows lost.")

analysis_df.head()

### 3.3. Structural Descriptors

Structural descriptors are derived from ground-truth annotations:

- Object density
- Occupied area ratio
- Presence of mosh pits
- Presence of raised hands
- Number of ground-truth objects

These variables aim to capture crowd-related and spatial complexity factors.

In [ ]:
# ==============================
# Compute structural descriptors
# ==============================

structural_data = []

# Resolve class names
names = dataset_config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
else:
    names = list(names)

images_for_analysis = set(analysis_df["image"].unique())

for img_path in test_images:

    if img_path.name not in images_for_analysis:
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    height, width = img.shape[:2]
    image_area = width * height

    label_path = test_labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path, "r") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()]

    num_objects = len(lines)
    total_box_area = 0.0

    presence_mosh = 0
    presence_hands = 0

    for line in lines:
        class_id, box = yolo_to_xyxy(line, width, height)

        box_area = max(0, (box[2] - box[0])) * max(0, (box[3] - box[1]))
        total_box_area += box_area

        class_name = names[class_id]

        if class_name == "mosh_pit":
            presence_mosh = 1

        if class_name == "hands_raised":
            presence_hands = 1

    density = num_objects / image_area if image_area > 0 else 0.0
    occupied_ratio = total_box_area / image_area if image_area > 0 else 0.0

    structural_data.append({
        "image": img_path.name,
        "density": density,
        "occupied_ratio": occupied_ratio,
        "presence_mosh_pit": presence_mosh,
        "presence_hands_raised": presence_hands
    })

structural_df = pd.DataFrame(structural_data)

print("Structural rows:", len(structural_df))
structural_df.head()

# ==============================
# Merge structural descriptors
# ==============================

analysis_df = analysis_df.merge(
    structural_df,
    on="image",
    how="inner"
)

print("Final analysis dataset size:", len(analysis_df))
analysis_df.head()

In [ ]:
# ==============================
# Save image-level analysis dataset
# ==============================

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "image_level_analysis.csv"
analysis_df.to_csv(output_path, index=False)

print("Image-level analysis dataset saved to:", output_path)
print("Total rows:", len(analysis_df))
print("Unique images:", analysis_df["image"].nunique())
print("Models present:", analysis_df["Model"].unique())
print("Columns:", list(analysis_df.columns))

## 4. Continuous Performance Analysis

### 4.1. Correlation Analysis

Pearson and Spearman correlation coefficients are computed between image-level recall and visual/structural descriptors.

This analysis aims to identify continuous relationships between scene properties and detection performance.

In [ ]:
# ==============================
# Correlation matrix (per model)
# ==============================

for model_label in analysis_df["Model"].unique():
    
    print(f"\n=== Correlation analysis for {model_label} model ===")
    
    df_model = analysis_df[analysis_df["Model"] == model_label]
    
    pearson_corr = df_model.corr(method="pearson", numeric_only=True)
    spearman_corr = df_model.corr(method="spearman", numeric_only=True)

    print("\nPearson correlation with recall:")
    print(pearson_corr["recall"].sort_values(ascending=False))

    print("\nSpearman correlation with recall:")
    print(spearman_corr["recall"].sort_values(ascending=False))

In [ ]:
corr_comparison = []

for model_label in analysis_df["Model"].unique():
    df_model = analysis_df[analysis_df["Model"] == model_label]
    pearson = df_model.corr(method="pearson", numeric_only=True)["recall"]
    
    corr_comparison.append({
        "Model": model_label,
        "corr_luminance": pearson.get("luminance_norm", None),
        "corr_density": pearson.get("density", None),
        "corr_blur": pearson.get("blur", None),
    })

corr_comparison_df = pd.DataFrame(corr_comparison)

display(corr_comparison_df)

### 4.2. Scatter Plots

Scatter plots are used to visualize the relationship between recall and selected descriptors, allowing inspection of linear and non-linear trends.

In [ ]:
# ==============================
# Scatter plots with regression + statistics (per model)
# ==============================

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def scatter_with_regression(x, y, xlabel, title, save_name):
    """
    Scatter plot with linear regression line,
    Pearson correlation coefficient and R².
    Automatically saves figure.
    """

    x = np.array(x)
    y = np.array(y)

    # Remove NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        print("Not enough points for regression.")
        return

    # Pearson correlation
    r, p_value = pearsonr(x, y)
    r2 = r**2

    # Linear regression
    coef = np.polyfit(x, y, 1)
    poly1d_fn = np.poly1d(coef)

    idx = np.argsort(x)
    x_sorted = x[idx]

    # Plot
    plt.figure(figsize=(6,4))
    plt.scatter(x, y)
    plt.plot(x_sorted, poly1d_fn(x_sorted))

    plt.xlabel(xlabel)
    plt.ylabel("Image-Level Recall")
    plt.title(title)

    textstr = (
        f"r = {r:.3f}\n"
        f"R² = {r2:.3f}\n"
        f"p = {p_value:.3f}"
    )

    plt.gca().text(
        0.05, 0.95,
        textstr,
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment='top'
    )

    plt.tight_layout()

    # SAVE
    plt.savefig(
        FIGURES_DIR / f"{save_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ==============================
# Generate plots per model
# ==============================

for model_label in analysis_df["Model"].unique():

    df_model = analysis_df[analysis_df["Model"] == model_label]

    print(f"\n=== Scatter plots for {model_label} model ===")

    scatter_with_regression(
        df_model["luminance_mean"],
        df_model["recall"],
        "Mean Luminance",
        f"{model_label} – Recall vs Luminance",
        f"{model_label}_recall_vs_luminance"
    )

    scatter_with_regression(
        df_model["blur"],
        df_model["recall"],
        "Blur (Variance of Laplacian)",
        f"{model_label} – Recall vs Blur",
        f"{model_label}_recall_vs_blur"
    )

    scatter_with_regression(
        df_model["density"],
        df_model["recall"],
        "Object Density",
        f"{model_label} – Recall vs Density",
        f"{model_label}_recall_vs_density"
    )

    scatter_with_regression(
        df_model["occupied_ratio"],
        df_model["recall"],
        "Occupied Area Ratio",
        f"{model_label} – Recall vs Occupied Area",
        f"{model_label}_recall_vs_occupied_ratio"
    )

print("Scatter plots exported to:", FIGURES_DIR)

In [ ]:
# ==========================================
# Interaction term analysis (per model)
# ==========================================

from scipy.stats import pearsonr
import numpy as np

TABLES_DIR.mkdir(parents=True, exist_ok=True)

interaction_results = []

for model_name in analysis_df["Model"].unique():

    df_model = analysis_df[analysis_df["Model"] == model_name].copy()

    # Interaction term: luminance × density
    interaction = df_model["density"] * df_model["luminance_mean"]

    # Remove NaNs safely
    mask = (
        ~df_model["recall"].isna() &
        ~interaction.isna()
    )

    recall_clean = df_model.loc[mask, "recall"]
    interaction_clean = interaction.loc[mask]

    if len(recall_clean) < 3:
        print(f"\nNot enough samples for interaction analysis: {model_name}")
        continue

    r, p = pearsonr(recall_clean, interaction_clean)

    interaction_results.append({
        "Model": model_name,
        "pearson_r": r,
        "r_squared": r**2,
        "p_value": p,
        "n_samples": len(recall_clean)
    })

    print(f"\n=== Interaction analysis for {model_name} model ===")
    print(f"Pearson r: {r:.4f}")
    print(f"R²: {r**2:.4f}")
    print(f"p-value: {p:.4f}")
    print(f"Samples: {len(recall_clean)}")


# ------------------------------------------
# Convert to DataFrame
# ------------------------------------------

interaction_df = pd.DataFrame(interaction_results)

display(interaction_df)


# ------------------------------------------
# Export
# ------------------------------------------

interaction_df.to_csv(
    TABLES_DIR / "interaction_term_analysis.csv",
    index=False
)

print("Interaction term analysis exported.")

In [ ]:
# ==========================================================
# EXPORT — Continuous Performance Analysis (Section 4)
# ==========================================================

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# 1. Export full image-level dataset used for analysis
# ----------------------------------------------------------

analysis_df.to_csv(
    TABLES_DIR / "image_level_analysis.csv",
    index=False
)

print("✔ image_level_analysis.csv exported")


# ----------------------------------------------------------
# 2. Export correlation comparison table
# ----------------------------------------------------------

corr_comparison_df.to_csv(
    TABLES_DIR / "correlation_comparison.csv",
    index=False
)

print("✔ correlation_comparison.csv exported")


# ----------------------------------------------------------
# 3. Export full Pearson & Spearman matrices (per model)
# ----------------------------------------------------------

for model_label in analysis_df["Model"].unique():

    df_model = analysis_df[analysis_df["Model"] == model_label]

    pearson_corr = df_model.corr(method="pearson", numeric_only=True)
    spearman_corr = df_model.corr(method="spearman", numeric_only=True)

    pearson_corr.to_csv(
        TABLES_DIR / f"{model_label}_pearson_correlation_matrix.csv"
    )

    spearman_corr.to_csv(
        TABLES_DIR / f"{model_label}_spearman_correlation_matrix.csv"
    )

print("✔ Correlation matrices exported")


# ----------------------------------------------------------
# 4. Export interaction term analysis
# ----------------------------------------------------------

interaction_df.to_csv(
    TABLES_DIR / "interaction_term_analysis.csv",
    index=False
)

print("✔ interaction_term_analysis.csv exported")


print("\n=== Continuous Performance Analysis exported successfully ===")
print("Files saved in:", TABLES_DIR)

### 4.3. Linear Regression

An Ordinary Least Squares (OLS) regression model is fitted to evaluate the joint effect of visual and structural descriptors on recall.

An interaction term between density and luminance is included to test whether illumination moderates crowd-related effects.

In [ ]:
# ==========================================================
# Final Linear Regression (HC3, simplified model)
# ==========================================================

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import pandas as pd

final_regression_results = []

for model_label in analysis_df["Model"].unique():

    print("\n=======================================")
    print(f"Final Linear Regression — {model_label}")
    print("=======================================\n")

    df_model = analysis_df[analysis_df["Model"] == model_label].copy()

    predictors = [
        "luminance_mean",
        "blur",
        "density",
        "presence_mosh_pit",
        "presence_hands_raised"
    ]

    X = df_model[predictors]
    y = df_model["recall"]

    mask = ~X.isnull().any(axis=1) & ~y.isnull()
    X = X[mask]
    y = y[mask]

    # Standardize continuous variables only
    continuous_vars = ["luminance_mean", "blur", "density"]

    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[continuous_vars] = scaler.fit_transform(X[continuous_vars])

    X_scaled = sm.add_constant(X_scaled)

    model = sm.OLS(y, X_scaled).fit(cov_type="HC3")

    print(model.summary())

    coef_df = pd.DataFrame({
        "Model": model_label,
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "p_value": model.pvalues.values
    })

    final_regression_results.append(coef_df)

# Export
final_regression_df = pd.concat(final_regression_results, ignore_index=True)

final_regression_df.to_csv(
    TABLES_DIR / "final_linear_regression_HC3.csv",
    index=False
)

print("\nFinal regression exported.")

In [ ]:
# ==========================================================
# Coefficient Plot — Pretrained vs Scratch (HC3)
# ==========================================================

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

coef_data = []

for model_label in analysis_df["Model"].unique():

    df_model = analysis_df[analysis_df["Model"] == model_label].copy()

    predictors = [
        "luminance_mean",
        "blur",
        "density",
        "presence_mosh_pit",
        "presence_hands_raised"
    ]

    X = df_model[predictors]
    y = df_model["recall"]

    mask = ~X.isnull().any(axis=1) & ~y.isnull()
    X = X[mask]
    y = y[mask]

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()

    continuous_vars = ["luminance_mean", "blur", "density"]
    X_scaled = X.copy()
    X_scaled[continuous_vars] = scaler.fit_transform(X[continuous_vars])

    import statsmodels.api as sm
    X_scaled = sm.add_constant(X_scaled)

    model = sm.OLS(y, X_scaled).fit(cov_type="HC3")

    conf_int = model.conf_int()

    for var in predictors:
        coef_data.append({
            "Model": model_label,
            "Variable": var,
            "Coefficient": model.params[var],
            "CI_low": conf_int.loc[var][0],
            "CI_high": conf_int.loc[var][1]
        })

coef_df = pd.DataFrame(coef_data)

# -----------------------------
# Plot
# -----------------------------

variables = predictors
y_positions = np.arange(len(variables))

fig, ax = plt.subplots(figsize=(8, 6))

for i, model_label in enumerate(coef_df["Model"].unique()):

    df_plot = coef_df[coef_df["Model"] == model_label]

    offset = -0.15 if model_label == "pretrained" else 0.15

    ax.errorbar(
        df_plot["Coefficient"],
        y_positions + offset,
        xerr=[
            df_plot["Coefficient"] - df_plot["CI_low"],
            df_plot["CI_high"] - df_plot["Coefficient"]
        ],
        fmt='o',
        label=model_label,
        capsize=4
    )

ax.axvline(0, linestyle='--')

ax.set_yticks(y_positions)
ax.set_yticklabels(variables)
ax.set_xlabel("Standardized Coefficient")
ax.set_title("Linear Regression Coefficients (HC3)")
ax.legend()

plt.tight_layout()

# Export figure
plt.savefig(FIGURES_DIR / "coefficient_plot_pretrained_vs_scratch.png", dpi=300)
plt.show()

print("Coefficient plot exported.")

### 4.4. Generalized Linear Model

Because recall is bounded between 0 and 1, a logistic GLM is fitted as an alternative specification.

This model tests whether the associations observed in OLS remain consistent under a bounded-response framework.

In [ ]:
# ==========================================================
# GLM (logit) — aligned with OLS model
# ==========================================================

import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

glm_results = []

for model_label in analysis_df["Model"].unique():

    print("\n=======================================")
    print(f"GLM (Logit) — {model_label}")
    print("=======================================\n")

    df_model = analysis_df[analysis_df["Model"] == model_label].copy()

    predictors = [
        "luminance_mean",
        "blur",
        "density",
        "presence_mosh_pit",
        "presence_hands_raised"
    ]

    X = df_model[predictors]
    y = df_model["recall"]

    mask = ~X.isnull().any(axis=1) & ~y.isnull()
    X = X[mask]
    y = y[mask]

    # -----------------------------------------
    # Standardize continuous predictors
    # -----------------------------------------

    continuous_vars = ["luminance_mean", "blur", "density"]

    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[continuous_vars] = scaler.fit_transform(X[continuous_vars])

    # -----------------------------------------
    # Ensure response strictly in (0,1)
    # -----------------------------------------

    eps = 1e-4
    y_beta = np.clip(y, eps, 1 - eps)

    # -----------------------------------------
    # Fit GLM (logit link)
    # -----------------------------------------

    X_scaled = sm.add_constant(X_scaled)

    model = sm.GLM(
        y_beta,
        X_scaled,
        family=sm.families.Binomial()
    ).fit(cov_type="HC3")

    print(model.summary())

    coef_df = pd.DataFrame({
        "Model": model_label,
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "p_value": model.pvalues.values
    })

    glm_results.append(coef_df)

# -----------------------------------------
# Export
# -----------------------------------------

glm_results_df = pd.concat(glm_results, ignore_index=True)

glm_results_df.to_csv(
    TABLES_DIR / "glm_logit_HC3.csv",
    index=False
)

print("\nGLM exported.")

## 5. Qualitative Error Analysis

This section presents visual inspection of best- and worst-performing test images.

The goal is to identify recurring failure patterns and compare qualitative observations with quantitative findings.

In [ ]:
# ==============================
# Qualitative Analysis — GT vs Predictions
# ==============================

import cv2
import matplotlib.pyplot as plt

def draw_boxes(image, boxes, color, label_prefix=""):
    """
    Draw bounding boxes on image.
    boxes: list of tuples (class_id, [x1,y1,x2,y2], score_optional)
    """
    img = image.copy()

    for item in boxes:
        if len(item) == 3:
            class_id, box, score = item
            text = f"{names[class_id]} ({score:.2f})"
        else:
            class_id, box = item
            text = f"{names[class_id]}"

        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(
            img,
            text,
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
            cv2.LINE_AA
        )

    return img


def visualize_image(img_name):

    img_path = test_image_path / img_name
    label_path = test_label_path / (Path(img_name).stem + ".txt")

    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ---- Load GT ----
    gt_boxes = []
    with open(label_path, "r") as f:
        for line in f:
            if line.strip():
                class_id, box = yolo_to_xyxy(line, img.shape[1], img.shape[0])
                gt_boxes.append((class_id, box))

    # ---- Predictions ----
    results_pre = models["pretrained"].predict(img_path, conf=0.25, verbose=False)
    results_scratch = models["scratch"].predict(img_path, conf=0.25, verbose=False)

    pred_pre = []
    pred_scratch = []

    if results_pre and results_pre[0].boxes is not None:
        for box in results_pre[0].boxes:
            pred_pre.append((
                int(box.cls[0]),
                box.xyxy[0].cpu().numpy(),
                float(box.conf[0])
            ))

    if results_scratch and results_scratch[0].boxes is not None:
        for box in results_scratch[0].boxes:
            pred_scratch.append((
                int(box.cls[0]),
                box.xyxy[0].cpu().numpy(),
                float(box.conf[0])
            ))

    # ---- Draw ----
    img_gt = draw_boxes(img_rgb, gt_boxes, (255,0,0))
    img_pre = draw_boxes(img_gt, pred_pre, (0,255,0))
    img_scratch = draw_boxes(img_gt, pred_scratch, (0,255,0))

    # ---- Plot ----
    fig, axes = plt.subplots(1,2, figsize=(14,6))

    axes[0].imshow(img_pre)
    axes[0].set_title("Pretrained Model")
    axes[0].axis("off")

    axes[1].imshow(img_scratch)
    axes[1].set_title("Scratch Model")
    axes[1].axis("off")

    plt.tight_layout()
    save_path = QUALITATIVE_DIR / f"{Path(img_name).stem}_comparison.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

### Best vs Worst Performing Images

Images are selected based on image-level recall to illustrate:

- Typical successful detections
- Typical failure cases
- Effects of occlusion, blur, and crowd density

In [ ]:
# ==============================
# Best and Worst Recall Images
# ==============================

# Ensure qualitative directory exists
QUALITATIVE_DIR.mkdir(parents=True, exist_ok=True)

# Sort images by recall
sorted_df = performance_df.sort_values("recall", ascending=False)

# Select top and bottom images
best_images = sorted_df["image"].unique()[:2]
worst_images = sorted_df.sort_values("recall")["image"].unique()[:2]

print("Best recall images:")
for img in best_images:
    print(f"Visualizing: {img}")
    visualize_image(img)

print("\nWorst recall images:")
for img in worst_images:
    print(f"Visualizing: {img}")
    visualize_image(img)

print("\nQualitative examples saved to:", QUALITATIVE_DIR)

## 6. Discussion

This evaluation provides both quantitative and structural insights into object detection performance in extreme concert environments.

### 1. Transfer Learning vs Training from Scratch

The pretrained YOLOv8s model clearly outperforms the model trained from scratch. The finetuned model achieves:
- mAP50-95 ≈ 0.21
- mAP50 ≈ 0.38
- Precision ≈ 0.38
- Recall ≈ 0.39

In comparison, the scratch model reaches:
- mAP50-95 ≈ 0.02
- Recall ≈ 0.10

The difference indicates that the dataset size is insufficient to train the detector from random initialization. The pretrained backbone, originally trained on COCO, provides feature representations that can be adapted to the EVOLVE dataset. Without this initialization, the model fails to learn meaningful detection patterns under the given data constraints.

The evaluation reveals structured performance variations across illumination and structural scene conditions.

### 2. Continuous Relationship Between Visual Factors and Recall

Beyond global metrics, image-level analysis reveals continuous relationships between recall and visual descriptors.

Correlation analysis for the pretrained model shows:
- A positive correlation between recall and mean luminance.
- Negative correlations between recall and:
    - presence of raised hands
    - presence of mosh pits
    - blur
    - object density

These relationships are not binary effects but continuous trends: as luminance increases, recall tends to increase; as crowd density or blur increases, recall tends to decrease.

Linear regression (R² ≈ 0.27) confirms that:
- Mean luminance is a significant positive predictor of recall.
- Presence of raised hands is a significant negative predictor.
- Presence of mosh pits shows a negative tendency.

The GLM model, adapted to the bounded nature of recall, leads to consistent conclusions.

Although the explained variance remains moderate, the convergence between correlation analysis, OLS, and GLM indicates that recall is partially structured by measurable image properties rather than random fluctuations.

### 3. Error Analysis

Qualitative inspection of best and worst recall images highlights recurrent failure modes.

In low-performing images, the model struggles with:
- Strong motion blur,
- Heavy occlusion (e.g., overlapping hands),
- High-density crowd regions,
- Small or partially visible objects.

Typical errors include:
- Missed detections in crowded regions,
- Confusion between overlapping objects,
- Incomplete bounding boxes in low-contrast areas.

In contrast, high-performing images tend to have:
- Higher luminance,
- Clear object boundaries,
- Lower crowd density,
- Less motion blur.

This qualitative analysis aligns with the quantitative findings from correlation and regression analyses.

### 4. Domain Shift

Concert images differ from standard object detection datasets in several ways:
- Low or uneven illumination,
- Strong color bias (red/blue lighting),
- High object density,
- Frequent occlusion,
- Motion blur.

The observed mAP values indicate that pretrained detectors can adapt partially to these conditions, but performance remains limited. This suggests that additional domain-specific data or training strategies may be required to improve detection accuracy in this setting

### 5. Limitations

Several limitations must be acknowledged:
- The dataset remains relatively small.
- Some classes (e.g., `mosh_pit`) have fewer instances.
- The test set contains 70 images.
- Only bounding box detection was evaluated.

Future work could involve increasing the dataset size, designing targeted augmentation strategies for low-light and blur conditions, or incorporating temporal information from video sequences.

## 7. Conclusion

This study evaluated object detection performance in extreme concert environments using a YOLOv8s architecture trained under two configurations: transfer learning (finetuning) and training from scratch.

The results show a clear performance gap between the two approaches. The finetuned model achieves moderate detection performance (mAP50-95 ≈ 0.21), whereas the scratch model fails to learn effective detection patterns under the current dataset size. This confirms that transfer learning is necessary when working with limited data in visually atypical domains.

Beyond global metrics, image-level analysis demonstrates that detection performance is not randomly distributed across images. Instead, recall exhibits measurable relationships with scene properties:
- It increases with mean luminance.
- It decreases in the presence of raised hands and mosh pits.
- It tends to decrease under higher blur and crowd density.

Regression and GLM analyses confirm that luminance and crowd-related features are statistically associated with recall variations. Although the explained variance remains moderate, the convergence of correlation, linear regression, and bounded-response modeling indicates that performance variability can be partially explained by measurable scene characteristics.

Qualitative inspection of high- and low-performing images further supports these findings. Failure cases are frequently associated with occlusion, dense crowds, motion blur, and low contrast conditions. This suggests that extreme concert environments introduce structural constraints that challenge standard object detectors.

Overall, this work shows that detection variability in volatile visual environments is not purely stochastic. It is partially structured by observable image properties. These findings provide a basis for future improvements, such as:
- Targeted augmentation strategies for low-light and blur conditions,
- Dataset expansion to better represent crowd dynamics,
- Investigation of architectures designed for dense or occluded scenes,
- Incorporation of temporal information from video sequences.

This project provides a domain-specific dataset (EVOLVE) and an evaluation procedure that links detection performance to measurable scene descriptors.